In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import pandas as pd
import json
import kagglehub
import os
import glob
import time

# 1. Cấu hình đường dẫn
# Tải dữ liệu
dataset_path = kagglehub.dataset_download("js042710/3tand1k")

# Định nghĩa thư mục đầu ra
OUTPUT_DIR = "cleaned_data"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# 2. Hàm xử lý làm sạch
def process_single_file(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                doc = json.loads(line)
                # Chỉ lấy các trường cần thiết để giảm bộ nhớ
                row = {
                    'user_id': doc['user_id'],
                    'timestamp': doc['timestamp'],
                    'recording_msid': doc['recording_msid'],
                    'track_name': doc['track_metadata']['track_name'],
                    'artist_name': doc['track_metadata']['artist_name']
                }
                data.append(row)
            except:
                continue
    
    if not data: return None

    df = pd.DataFrame(data)
    
    # --- LOGIC LÀM SẠCH ---
    df = df.sort_values(by=['user_id', 'timestamp'])
    
    # Tính toán để lọc spam (nghe < 30s cùng 1 bài)
    df['prev_timestamp'] = df.groupby('user_id')['timestamp'].shift(1)
    df['prev_msid'] = df.groupby('user_id')['recording_msid'].shift(1)
    df['time_diff'] = df['timestamp'] - df['prev_timestamp']
    
    # Điều kiện lọc
    condition_spam = (df['recording_msid'] == df['prev_msid']) & (df['time_diff'] < 30)
    df_clean = df[~condition_spam].copy()
    
    # Xóa cột tạm & cột không cần thiết cho output
    df_clean = df_clean.drop(columns=['prev_timestamp', 'prev_msid', 'time_diff'])
    
    return df_clean

# 3. VÒNG LẶP CHÍNH (Main Loop)
# Tìm tất cả file .listens trong mọi thư mục con
all_files = glob.glob(os.path.join(dataset_path, "**/*.listens"), recursive=True)

print(f"Tổng số file tìm thấy: {len(all_files)}")

for i, input_path in enumerate(all_files):
    # Tạo tên file đầu ra tương ứng
    file_name = os.path.basename(input_path) # ví dụ: 15.1.2026.listens
    output_name = f"clean_{file_name}.csv"   # ví dụ: clean_15.1.2026.listens.csv
    output_path = os.path.join(OUTPUT_DIR, output_name)
    
    # --- KIỂM TRA QUAN TRỌNG ---
    if os.path.exists(output_path):
        print(f"[{i+1}/{len(all_files)}] Đã tồn tại, bỏ qua: {file_name}")
        continue # Nhảy qua file này, không làm gì cả
    
    # Nếu chưa tồn tại thì mới xử lý
    print(f"[{i+1}/{len(all_files)}] Đang xử lý: {file_name}...")
    start_time = time.time()
    
    try:
        clean_df = process_single_file(input_path)
        
        if clean_df is not None and not clean_df.empty:
            # Lưu file CSV (hoặc Parquet nếu dữ liệu lớn)
            clean_df.to_csv(output_path, index=False, encoding='utf-8')
            print(f"   -> Đã lưu: {len(clean_df)} dòng ({time.time() - start_time:.2f}s)")
        else:
            print(f"   -> File rỗng hoặc lỗi.")
            
    except Exception as e:
        print(f"   -> LỖI: {e}")

print("\nHoàn tất quá trình làm sạch!")

Tổng số file tìm thấy: 30
[1/30] Đang xử lý: 1.2.2026.listens...
   -> Đã lưu: 3299722 dòng (53.30s)
[2/30] Đang xử lý: 8.1.2026.listens...
   -> Đã lưu: 2405492 dòng (37.35s)
[3/30] Đang xử lý: 17.1.2026.listens...
   -> Đã lưu: 3038158 dòng (46.77s)
[4/30] Đang xử lý: 10.1.2026.listens...
   -> Đã lưu: 2815506 dòng (45.91s)
[5/30] Đang xử lý: 24.1.2026.listens...
   -> Đã lưu: 2084304 dòng (34.53s)
[6/30] Đang xử lý: 29.1.2026.listens...
   -> Đã lưu: 2141650 dòng (33.80s)
[7/30] Đang xử lý: 13.1.2026.listens...
   -> Đã lưu: 2434666 dòng (38.05s)
[8/30] Đang xử lý: 19.1.2026.listens...
   -> Đã lưu: 2300483 dòng (37.26s)
[9/30] Đang xử lý: 9.1.2026.listens...
   -> Đã lưu: 3250328 dòng (50.59s)
[10/30] Đang xử lý: 2.2.2026.listens...
   -> Đã lưu: 2998531 dòng (47.14s)
[11/30] Đang xử lý: 30.1.2026.listens...
   -> Đã lưu: 3258578 dòng (54.86s)
[12/30] Đang xử lý: 27.1.2026.listens...
   -> Đã lưu: 3736039 dòng (59.79s)
[13/30] Đang xử lý: 15.1.2026.listens...
   -> Đã lưu: 2600356 

In [3]:
import shutil
from IPython.display import FileLink

# 1. Nén toàn bộ thư mục 'cleaned_data' thành file 'cleaned_data.zip'
shutil.make_archive('cleaned_data', 'zip', 'cleaned_data')

# 2. Tạo đường dẫn tải xuống ngay trong Output của Cell
print("Đã nén xong! Click vào link bên dưới để tải về:")
FileLink(r'cleaned_data.zip')

Đã nén xong! Click vào link bên dưới để tải về:


/kaggle/working/cleaned_data.zip